# 🔎 Data Exploration with Pandas
## Module Data Science

> **Prérequis :** NumPy, File Handling, Data Cleaning with Pandas

---

Notebook **interactif** utilisant le vrai jeu de données **`bootcamp_500.csv`** (510 lignes).

⚠️ **Avant de commencer** : téléchargez `bootcamp_500.csv` dans Colab (panneau Fichiers à gauche, bouton "Importer").

### Table des matières
1. Introduction
2. Initial Data Analysis
3. Handling Missing Data
4. Outliers
5. Data Anomalies
6. Data Encoding
7. Exploratory Data Analysis (EDA)
8. Data Pre-processing Checkpoint (projet)
9. Exercices pratiques


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Charger le jeu de données (510 lignes)
df = pd.read_csv("bootcamp_500.csv")
print("Dimensions :", df.shape)
df.head()

---
## 1. Introduction to Data Exploration

L'exploration = **comprendre et diagnostiquer** les données AVANT de les analyser/modéliser.

```
COLLECTE → EXPLORATION → NETTOYAGE → PRÉPARATION → MODÉLISATION
```

---
## 2. Initial Data Analysis with Pandas

Les premiers réflexes face à un nouveau jeu de données.

In [ ]:
print("Shape :", df.shape)
df.info()

In [ ]:
# describe() révèle déjà les outliers (age min=-5, max=200 ; note max=30 !)
df.describe().round(2)

In [ ]:
# Variables catégorielles : value_counts révèle les anomalies de casse
print(df["ville"].value_counts())

In [ ]:
print("Niveaux :", df["niveau"].unique())
print("\nProportions (%) :")
print((df["niveau"].value_counts(normalize=True) * 100).round(1))

---
## 3. Handling Missing Data — Valeurs manquantes

In [ ]:
# Détecter
print("Manquants par colonne :")
print(df.isna().sum())
print("\nPourcentage :")
print((df.isna().sum() / len(df) * 100).round(2))

In [ ]:
# Imputer : médiane (numérique), mode (catégoriel)
df_imp = df.copy()
df_imp["note_python"] = df_imp["note_python"].fillna(df_imp["note_python"].median())
df_imp["ville"] = df_imp["ville"].fillna(df_imp["ville"].mode()[0])
print("Manquants après imputation partielle :", df_imp[["note_python","ville"]].isna().sum().sum())

---
## 4. Outliers — Valeurs aberrantes

In [ ]:
# Boxplot — les outliers apparaissent comme des points isolés
df.boxplot(column=["age"])
plt.title("Détection d'outliers sur l'âge")
plt.show()

In [ ]:
# Méthode IQR
Q1 = df["age"].quantile(0.25)
Q3 = df["age"].quantile(0.75)
IQR = Q3 - Q1
bb = Q1 - 1.5 * IQR
bh = Q3 + 1.5 * IQR
print(f"Q1={Q1}, Q3={Q3}, IQR={IQR}")
print(f"Bornes normales : [{bb}, {bh}]")

outliers = df[(df["age"] < bb) | (df["age"] > bh)]
print("\nÂges aberrants détectés :", sorted(outliers["age"].tolist()))

In [ ]:
# Z-score
serie = df["age"]
z = (serie - serie.mean()) / serie.std()
print("Outliers |z| > 3 :", sorted(df[z.abs() > 3]["age"].tolist()))

---
## 5. Data Anomalies — Anomalies de données

In [ ]:
# Anomalies de casse sur les villes
print("Villes AVANT nettoyage :", df["ville"].nunique(), "valeurs uniques")
villes_propres = df["ville"].str.strip().str.title()
print("Villes APRÈS nettoyage :", villes_propres.nunique(), "valeurs uniques")
print("\nAprès nettoyage :")
print(villes_propres.value_counts())

In [ ]:
# Valeurs impossibles (règles métier)
print("Âges impossibles (<15 ou >100) :", ((df["age"]<15)|(df["age"]>100)).sum())
print("Notes SQL impossibles (>20)     :", (df["note_sql"]>20).sum())

---
## 6. Data Encoding — Encodage des variables

- **Ordinal** (avec ordre) → Label Encoding
- **Nominal** (sans ordre) → One-Hot Encoding

In [ ]:
# niveau = ORDINAL → Label Encoding
ordre = {"Débutant": 0, "Intermédiaire": 1, "Avancé": 2}
df_enc = df.copy()
df_enc["niveau_encode"] = df_enc["niveau"].map(ordre)
print(df_enc[["niveau", "niveau_encode"]].head())

In [ ]:
# ville = NOMINAL → One-Hot Encoding
df_enc["ville"] = df_enc["ville"].str.strip().str.title()
df_onehot = pd.get_dummies(df_enc, columns=["ville"], prefix="ville")
print("Colonnes après One-Hot :", [c for c in df_onehot.columns if c.startswith("ville_")])

---
## 7. Exploratory Data Analysis (EDA)

Sur 510 lignes, les distributions et corrélations deviennent vraiment parlantes.

In [ ]:
# Histogramme (distribution en cloche visible sur 510 lignes)
df["note_sql"].hist(bins=20)
plt.title("Distribution des notes SQL")
plt.xlabel("Note"); plt.ylabel("Fréquence")
plt.show()

In [ ]:
# Moyenne des notes par niveau
print(df.groupby("niveau")["note_sql"].mean().round(2))

In [ ]:
# Nuage de points : relation heures d'étude / note
df_v = df[(df["note_sql"] <= 20)]  # exclure les notes impossibles
df_v.plot(kind="scatter", x="heures_etude", y="note_sql", alpha=0.5)
plt.title("Relation heures d'étude / note SQL")
plt.show()

In [ ]:
# Matrice de corrélation (sur données valides)
df_valide = df[(df["age"]>=15)&(df["age"]<=100)&(df["note_sql"]<=20)]
correlation = df_valide[["age","heures_etude","note_sql","note_python","salaire_stage"]].corr()
print(correlation.round(2))

---
## 8. Data Pre-processing Checkpoint — Projet

Pipeline complet sur `bootcamp_500.csv` : exploration → anomalies → valeurs impossibles → imputation → doublons → encodage.

In [ ]:
# ÉTAPE 0 : charger
df = pd.read_csv("bootcamp_500.csv")
print("Brut :", df.shape)

In [ ]:
# ÉTAPE 1 : exploration
print("Manquants :\n", df.isna().sum())
print("\nStats :\n", df.describe().round(2))

In [ ]:
# ÉTAPE 2 : anomalies (casse/espaces des villes)
print("Villes avant :", df["ville"].nunique())
df["ville"] = df["ville"].str.strip().str.title()
print("Villes après :", df["ville"].nunique())

In [ ]:
# ÉTAPE 3 : valeurs impossibles → NaN
df.loc[(df["age"] < 15) | (df["age"] > 100), "age"] = np.nan
df.loc[df["note_sql"] > 20, "note_sql"] = np.nan
print("Après mise à NaN des valeurs impossibles :")
print(df[["age","note_sql"]].isna().sum())

In [ ]:
# ÉTAPE 4 : imputation (médiane + mode)
df["age"]         = df["age"].fillna(df["age"].median())
df["note_sql"]    = df["note_sql"].fillna(df["note_sql"].median())
df["note_python"] = df["note_python"].fillna(df["note_python"].median())
df["ville"]       = df["ville"].fillna(df["ville"].mode()[0])
print("Manquants restants :", df.isna().sum().sum())

In [ ]:
# ÉTAPE 5 : doublons
avant = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Doublons supprimés : {avant - len(df)}")

In [ ]:
# ÉTAPE 6 : encodage
ordre = {"Débutant": 0, "Intermédiaire": 1, "Avancé": 2}
df["niveau_encode"] = df["niveau"].map(ordre)
df_final = pd.get_dummies(df, columns=["ville"], prefix="ville")
print("Dimensions finales :", df_final.shape)

In [ ]:
# ÉTAPE 7 : sauvegarde
df_final.to_csv("bootcamp_500_pretraite.csv", index=False, encoding="utf-8")
print("✅ Sauvegardé dans bootcamp_500_pretraite.csv")
df_final.head()

---
## 9. 💻 Exercices pratiques

Complétez chaque cellule "✏️ À vous de jouer !", puis comparez avec la solution.

### Exercice 1 — Analyse initiale
Rechargez `bootcamp_500.csv` et affichez dimensions, types, valeurs manquantes et statistiques.

In [ ]:
# ✏️ À vous de jouer !



In [ ]:
# ✅ Solution Exercice 1
df = pd.read_csv("bootcamp_500.csv")
print("Dimensions :", df.shape)
print("\nTypes :\n", df.dtypes)
print("\nManquants :\n", df.isna().sum())
print("\nStats :\n", df.describe().round(2))

### Exercice 2 — Valeurs manquantes
Calculez le % de manquants de `note_sql` et imputez par la médiane.

In [ ]:
# ✏️ À vous de jouer !



In [ ]:
# ✅ Solution Exercice 2
pct = df["note_sql"].isna().sum() / len(df) * 100
print(f"Manquants note_sql : {pct:.1f}%")
df["note_sql"] = df["note_sql"].fillna(df["note_sql"].median())
print("Après imputation :", df["note_sql"].isna().sum())

### Exercice 3 — Outliers (IQR)
Détectez les outliers de la colonne `age` avec la méthode IQR.

In [ ]:
# ✏️ À vous de jouer !



In [ ]:
# ✅ Solution Exercice 3
df = pd.read_csv("bootcamp_500.csv")
Q1, Q3 = df["age"].quantile(0.25), df["age"].quantile(0.75)
IQR = Q3 - Q1
bb, bh = Q1 - 1.5*IQR, Q3 + 1.5*IQR
print("Bornes :", bb, bh)
print("Outliers :", sorted(df[(df["age"]<bb)|(df["age"]>bh)]["age"].tolist()))

### Exercice 4 — Anomalies
Uniformisez la casse/espaces de la colonne `ville` et comptez les valeurs uniques avant/après.

In [ ]:
# ✏️ À vous de jouer !



In [ ]:
# ✅ Solution Exercice 4
print("Avant :", df["ville"].nunique())
df["ville"] = df["ville"].str.strip().str.title()
print("Après :", df["ville"].nunique())
print(df["ville"].value_counts())

### Exercice 5 — Encodage
Encodez `niveau` (ordinal, Label) et `ville` (nominal, One-Hot).

In [ ]:
# ✏️ À vous de jouer !



In [ ]:
# ✅ Solution Exercice 5
ordre = {"Débutant": 0, "Intermédiaire": 1, "Avancé": 2}
df["niveau_encode"] = df["niveau"].map(ordre)
df_enc = pd.get_dummies(df, columns=["ville"], prefix="ville")
print(df_enc[["niveau","niveau_encode"]].head())
print("Colonnes ville_* :", [c for c in df_enc.columns if c.startswith("ville_")])

### Exercice 6 — Corrélation
Calculez la matrice de corrélation des variables numériques (sur données valides).

In [ ]:
# ✏️ À vous de jouer !



In [ ]:
# ✅ Solution Exercice 6
dv = df[(df["age"]>=15)&(df["age"]<=100)&(df["note_sql"]<=20)]
print(dv[["age","heures_etude","note_sql","note_python","salaire_stage"]].corr().round(2))
# heures_etude ↔ notes et note_sql ↔ note_python sont fortement corrélées

### 🏆 Challenge bonus
Repartez de `bootcamp_500_pretraite.csv` (issu du projet) et répondez :
1. Ville la plus représentée
2. Moyenne des notes SQL par niveau
3. Corrélation heures_etude / note_python
4. Histogramme des notes + boxplot des salaires
5. Nombre d'étudiants avec moyenne >= 14

In [ ]:
# ✏️ À vous de jouer !



In [ ]:
# ✅ Piste de solution Challenge
df = pd.read_csv("bootcamp_500.csv")
df["ville"] = df["ville"].str.strip().str.title()
dv = df[(df["note_python"].notna()) & (df["note_sql"] <= 20)].copy()

print("1. Ville la + fréquente :", df["ville"].value_counts().idxmax())
print("2. Moyenne SQL par niveau :\n", df.groupby("niveau")["note_sql"].mean().round(2))
print("3. Corrélation heures/python :", dv["heures_etude"].corr(dv["note_python"]).round(2))

dv["note_sql"].hist(bins=20); plt.title("Notes SQL"); plt.show()
dv.boxplot(column=["salaire_stage"]); plt.title("Salaires"); plt.show()

dv["moyenne"] = (dv["note_sql"] + dv["note_python"]) / 2
print("5. Étudiants moyenne >= 14 :", (dv["moyenne"] >= 14).sum())

---
## 🎉 Félicitations !

Vous maîtrisez l'exploration de données sur un vrai jeu de 510 lignes :
- **Analyse initiale** — info(), describe(), value_counts()
- **Valeurs manquantes** — détection et imputation
- **Outliers** — boxplot, IQR, Z-score
- **Anomalies** — casse, valeurs impossibles
- **Encodage** — Label vs One-Hot
- **EDA** — distributions, corrélations

**Prochaine étape :** Pandas Profiling (automatiser tout ça), puis Machine Learning.

*📘 Module Data Science — Data Exploration with Pandas | Bootcamp Data Science*